In [1]:
from collections import defaultdict
from itertools import product
import json
import numpy as np
from sentence_transformers import SentenceTransformer, util
import statistics
import torch

from dataset import  CDCR_Dataset

#Load data
ecb_paths = ['../CDCR/data/ECBplus-prep/output_data/train' ,'../CDCR/data/NewsWCL50-prep/output_data/train']
cdcr_train = CDCR_Dataset(name="newswcl50_train", data_path=ecb_paths,context_method="hsu")

# Load SBERT model
model = SentenceTransformer('all-mpnet-base-v2')

c:\ProgramData\miniconda3\envs\mbib\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# list of sentences by topic

In [8]:
cdcr_train.mentions_by_sentence.keys()

dict_keys(['1/1ecb/10/3', '1/1ecb/11/0', '1/1ecb/14/1', '1/1ecb/15/1', '1/1ecb/5/3', '1/1ecb/7/1', '1/1ecb/8/1', '1/1ecb/10/0', '1/1ecb/11/2', '1/1ecb/14/2', '1/1ecb/17/2', '1/1ecb/18/0', '1/1ecb/19/3', '1/1ecb/1/2', '1/1ecb/2/0', '1/1ecb/2/2', '1/1ecb/3/1', '1/1ecb/4/2', '1/1ecb/5/1', '1/1ecb/5/2', '1/1ecb/6/1', '1/1ecb/6/0', '1/1ecb/7/4', '1/1ecb/1/1', '1/1ecb/12/1', '1/1ecb/13/1', '1/1ecb/14/5', '1/1ecb/14/0', '1/1ecb/15/0', '1/1ecb/17/1', '1/1ecb/17/0', '1/1ecb/19/1', '1/1ecb/19/0', '1/1ecb/4/1', '1/1ecb/7/2', '1/1ecb/7/5', '1/1ecb/7/0', '1/1ecb/9/4', '1/1ecb/9/3', '1/1ecbplus/10/3', '1/1ecbplus/13/3', '1/1ecbplus/13/1', '1/1ecbplus/14/4', '1/1ecbplus/1/4', '1/1ecbplus/7/3', '1/1ecbplus/10/1', '1/1ecbplus/11/5', '1/1ecbplus/11/2', '1/1ecbplus/11/1', '1/1ecbplus/14/1', '1/1ecbplus/15/3', '1/1ecbplus/15/1', '1/1ecbplus/17/1', '1/1ecbplus/17/4', '1/1ecbplus/19/3', '1/1ecbplus/19/4', '1/1ecbplus/19/1', '1/1ecbplus/1/3', '1/1ecbplus/1/1', '1/1ecbplus/20/3', '1/1ecbplus/20/1', '1/1ecbplu

In [156]:
#TAKES TOO LONG, LOAD RESULTS INSTEAD

def return_sentence_pair_similarity_by_topic(dict_sentences_by_topic):
    
    dict_sentences_pair_similarity_by_topic = {}
    for topic,sentences_dict in dict_sentences_by_topic.items(): #Now this come from the dataset
        print(f'current topic {topic}')
        sentences = list(sentences_dict.values())
        list_of_keys = list(sentences_dict.keys())

        #Compute similarity
        if not sentences:
            print(f"topic {topic} is empty")
            continue

        sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
        cosine_scores = util.cos_sim(sentence_embeddings, sentence_embeddings)

        # Get the indices of the lower triangular elements of the similarity matrix
        indices = torch.tril(torch.ones_like(cosine_scores), diagonal=-1).nonzero()

        # Create a dictionary using the indices
        lower_triangle_dict = {}

        for i, j in indices:
            key_sentence1 = list_of_keys[i.item()]
            key_sentence2 = list_of_keys[j.item()]
            key_pair = (key_sentence1, key_sentence2) 

            #Ignore sentences without mentions
            if not cdcr_train.mentions_by_sentence[key_sentence1] or not cdcr_train.mentions_by_sentence[key_sentence2]:
                continue

            similarity_score = round(cosine_scores[i, j].item(), 3)
            if similarity_score<1: #ignores those sentences with similarity=1. Those are mostly exact citations
                lower_triangle_dict[key_pair] = similarity_score

        dict_sentences_pair_similarity_by_topic[topic] = lower_triangle_dict

        return dict_sentences_pair_similarity_by_topic

def unnest_dictionary(input_dict,key=''):
  
    unnested_dict = {}
    for key, value in input_dict.items():
        if isinstance(value, dict):
            unnested_dict.update(unnest_dictionary(value, key))
        else:
            unnested_dict[key] = value

    return unnested_dict

def create_pairs(dict_sentences_pairs_similarity,mentions,mentions_by_sentence, mention_type):
    dict_positive_pairs_by_topic = defaultdict(dict)
    dict_negative_pairs_by_topic = defaultdict(dict)

    for topic, sentence_pairs in dict_sentences_pairs_similarity.items():

        if not sentence_pairs: 
            print(f"topic {topic} does not have sentence pairs")
            continue
        
        topic_sentence_similarity_median = statistics.median(list(sentence_pairs.values()))

        for (sentence_key1, sentence_key2), score in sentence_pairs.items():
            mentions_sentence1 = mentions_by_sentence[sentence_key1]
            mentions_sentence2 = mentions_by_sentence[sentence_key2]

            #Check the combination of all mentions for each pair of sentences
            for mention_key1,mention_key2 in product(mentions_sentence1, mentions_sentence2):

                mention1 = mentions[mention_key1]
                mention2 = mentions[mention_key2]

                if mention1.type!=mention_type or mention2.type!=mention_type: # No cross-type comparison
                    continue

                same_coref_chain = mention1.coref_chain == mention2.coref_chain
                new_dict_entry = {(mention_key1,mention_key2):score}

                if same_coref_chain:
                    dict_positive_pairs_by_topic[topic].update(new_dict_entry)
                else:
                    dict_negative_pairs_by_topic[topic].update(new_dict_entry)

            
        #Subsampling
        positive_pairs_scores= list(dict_positive_pairs_by_topic[topic].values())

        #Downsample negative pairs based on positive pairs score's Q3
        third_quartile = np.percentile(positive_pairs_scores, 75)
        dict_negative_pairs_by_topic[topic] = {key: value for key, value in dict_negative_pairs_by_topic[topic].items() if value > third_quartile}
        
        #Downsample positive pairs based on topic's median
        dict_positive_pairs_by_topic[topic] = {key: value for key, value in dict_positive_pairs_by_topic[topic].items() if value < topic_sentence_similarity_median}
    
    return unnest_dictionary(dict_positive_pairs_by_topic), unnest_dictionary(dict_negative_pairs_by_topic)


def return_pairs_and_labels (dict_sentences_by_topic, mentions, mentions_by_sentence, mention_type):
    sentences_pairs_similarity_dict = return_sentence_pair_similarity_by_topic(dict_sentences_by_topic) #Dict sentence by 
    
    positive_pairs,negative_pairs = create_pairs(dict_sentences_pairs_similarity=sentences_pairs_similarity_dict,
                                    mentions=mentions,mentions_by_sentence=mentions_by_sentence, mention_type=mention_type)

    positive_labels, negative_labels= [1]*len(positive_pairs), [0]*len(negative_pairs)

    all_pairs = positive_pairs + negative_pairs
    all_labels = positive_labels + negative_labels

    return all_pairs,all_labels



def load_sample_keys_and_labels(self, pairs_path,label):

    with open(pairs_path,"r") as json_file:
        pairs = json.load(json_file)[self.mention_type]
    
    labels = [label]*len(pairs)
    return list(pairs.keys()), labels

current topic 1
current topic 10
current topic 11
current topic 13
current topic 14
current topic 16
current topic 18
current topic 19
current topic 20
current topic 21
current topic 22
current topic 23
current topic 24
current topic 25
current topic 26
current topic 27
current topic 28
current topic 29
current topic 3
current topic 30
current topic 31
current topic 32
current topic 33
current topic 36
current topic 37
current topic 38
current topic 39
current topic 4
current topic 41
current topic 43
current topic 44
current topic 6
current topic 7
current topic 8
current topic 9


In [213]:
# Export results
# Convert tuple keys to strings
def convert_keys_to_strings(d):
    new_dict = {}
    for key, value in d.items():
        if isinstance(key, tuple):
            key = '###'.join(map(str, key))  # Convert tuple to a string
        if isinstance(value, dict):
            value = convert_keys_to_strings(value)  # Recursively convert nested dictionaries
        new_dict[key] = value
    return new_dict

nested_dict_strings = convert_keys_to_strings(dict_sentences_pair_by_topic)


json_file_path = 'sentence_similarities.json'

# Export the nested dictionary to JSON
with open(json_file_path, 'w') as json_file:
    json.dump(nested_dict_strings, json_file, indent=4)

print(f'Nested dictionary has been exported to {json_file_path}.')

Nested dictionary has been exported to sentence_similarities.json.


In [159]:
# json_file_path = 'sentence_similarities.json'
# with open(json_file_path, 'r') as json_file:
#     nested_dict_strings = json.load(json_file)

# dict_sentences_pair_by_topic = {}

# for key, inner_dict in nested_dict_strings.items():
#     new_inner_dict = {tuple(k.split('###')): v for k, v in inner_dict.items()}
#     dict_sentences_pair_by_topic[key] = new_inner_dict

## Extract mentions from sentences

In [170]:
len(negative_pairs['event'])

17567

In [214]:
positive_pairs_entities,negative_pairs_entities = create_pairs(dict_sentences_pairs=dict_sentences_pair_by_topic,
                                 cdcr_dataset=cdcr_train, mention_type='entity')

positive_pairs_events,negative_pairs_events = create_pairs(dict_sentences_pairs=dict_sentences_pair_by_topic,
                                 cdcr_dataset=cdcr_train, mention_type='event')

positive_pairs = {'entity':positive_pairs_entities, 'event':positive_pairs_events}
negative_pairs = {'entity':negative_pairs_entities,'event':negative_pairs_events}

positive_pairs_export = convert_keys_to_strings(positive_pairs)
negative_pairs_export = convert_keys_to_strings(negative_pairs)

In [215]:
#Export definite pairs
json_file_path = "positive_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(positive_pairs_export, json_file, indent=4)

json_file_path = "negative_pairs.json"

# Export dictionary to JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(negative_pairs_export, json_file, indent=4)

In [182]:
## Statistics from pairs

cdcr_train.mentions['9_9ecbplus_NON17210019876807755_1_9_YFNM'].subtopic_id

'9ecbplus'

In [196]:
## Statistics from pairs
count_same_topic = 0
count_different_topic = 0
entity_type = 'event'
for (mention1_key, mention2_key), value in negative_pairs[entity_type].items() | positive_pairs[entity_type].items():
    subtopic1 = cdcr_train.mentions[mention1_key].subtopic_id
    subtopic2 = cdcr_train.mentions[mention2_key].subtopic_id
    if subtopic1 ==subtopic2:
        count_same_topic+=1
    else:
        count_different_topic+=1
    
print(count_same_topic/(count_same_topic+count_different_topic))

0.9465135278004709


In [202]:
print('positive similarity score entities', np.median(list(positive_pairs['entity'].values())))
print('positive similarity score events', np.median(list(positive_pairs['event'].values())))
print('negative similarity score entities', np.median(list(negative_pairs['entity'].values())))
print('negative similarity score events', np.median(list(negative_pairs['event'].values())))

positive similarity score entities 0.379
positive similarity score events 0.366
negative similarity score entities 0.803
negative similarity score events 0.812
